In [1]:
from qdrant_client import QdrantClient, AsyncQdrantClient
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.text_embeddings_inference import TextEmbeddingsInference
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from pathlib import Path
import numpy as np
import torch
import gc
import re
import pandas as pd
from typing import Dict, List, Any
import asyncio
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json

/home/yuri/HSE/ВКР/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calculate_metrics(
    retrieved_nodes: List[Any],
    relevant_pages: List[str] | set,
    k: int = 5
) -> Dict[str, float]:
    """
    Подсчёт метрик качества поиска для ОДНОГО запроса.
    
    Метрики:
    - precision@k: доля релевантных чанков среди top-k
    - recall@k: доля найденных релевантных страниц
    - f1@k: гармоническое среднее precision и recall
    - hit_rate@k: 1.0 если хотя бы один чанк в top-k релевантен, иначе 0.0
    - mrr@k: 1 / ранг первого релевантного чанка (0.0 если не найден)
    """
    relevant_pages = set(relevant_pages)
    
    retrieved_pages = set()
    relevant_chunks_count = 0
    first_relevant_rank = 0  # для MRR
    
    for i, node in enumerate(retrieved_nodes[:k]):
        chunk_pages = set(node.metadata.get("pages_covered", []))
        overlap = chunk_pages & relevant_pages
        
        retrieved_pages.update(chunk_pages)
        
        if overlap:
            relevant_chunks_count += 1
            if first_relevant_rank == 0:
                first_relevant_rank = i + 1  # ранг (начиная с 1)
    
    # Precision: доля релевантных чанков в top-k
    precision = relevant_chunks_count / k if k > 0 else 0.0
    
    # Recall: доля релевантных страниц, которые удалось найти
    recall = (
        len(retrieved_pages & relevant_pages) / len(relevant_pages)
        if relevant_pages else 0.0
    )
    
    # F1
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )
    
    # Hit Rate (бинарный): нашёлся ли хотя бы 1 релевантный чанк
    hit_rate = 1.0 if relevant_chunks_count > 0 else 0.0
    
    # MRR: обратный ранг первого релевантного чанка
    mrr = 1.0 / first_relevant_rank if first_relevant_rank > 0 else 0.0
    
    return {
        "precision@k": round(precision, 4),
        "recall@k": round(recall, 4),
        "f1@k": round(f1, 4),
        "hit_rate@k": round(hit_rate, 4),
        "mrr@k": round(mrr, 4),
    }

In [3]:


# def evaluate_collection(
#     index,
#     eval_data: List[Dict],
#     top_k: int = 5,
#     retrieval_k: int = 10
# ) -> pd.DataFrame:
#     """
#     Полная оценка ОДНОЙ коллекции (одного chunk_size) по всему набору тестовых запросов.
    
#     Входные параметры:
#     - index: готовый VectorStoreIndex из LlamaIndex (созданный для текущей коллекции)
#     - eval_data (List[Dict]): список словарей с тестовыми данными.
#                               Каждый элемент должен иметь ровно два ключа:
#                               {
#                                   "query": str,                    # текст вопроса
#                                   "relevant_pages": List[str] | set  # релевантные страницы (как в calculate_metrics)
#                               }
#                               Пример: [{"query": "Какой срок оплаты?", "relevant_pages": ["12", "13"]}, ...]
#     - top_k (int, по умолчанию 5): на каком @k считать метрики (precision@5, hit_rate@5 и т.д.)
#     - retrieval_k (int, по умолчанию 10): сколько чанков запрашивать у retriever (обычно больше top_k, чтобы было из чего выбирать)
    
#     Выходные параметры:
#     - pd.DataFrame: таблица, где каждая строка — результаты по одному запросу (все метрики).
#                     В конце функции также печатается строка со средними значениями по всем запросам.
#                     Колонки DataFrame: precision@k, recall@k, f1@k, hit_rate@k, ndcg@k
#     """
#     all_metrics = []
    
#     for item in eval_data:
#         retriever = index.as_retriever(similarity_top_k=retrieval_k)
#         nodes = retriever.retrieve(item["query"])
        
#         metrics = calculate_metrics(nodes, item["relevant_pages"], k=top_k)
#         all_metrics.append(metrics)
    
#     df = pd.DataFrame(all_metrics)
#     avg = df.mean().round(4)
    
#     # print(f"\n=== Средние метрики по {len(eval_data)} запросам (top@{top_k}) ===")
#     # print(avg)
    
#     return avg

In [4]:
async def evaluate_collection(
    index,
    eval_data: List[Dict],
    top_k: int = 5,
    retrieval_k: int = 10
) -> pd.Series:
    retriever = index.as_retriever(similarity_top_k=retrieval_k)
    
    tasks = [retriever.aretrieve(item["query"]) for item in eval_data]
    all_retrieved_nodes = await asyncio.gather(*tasks)
    
    all_metrics = []
    for nodes, item in zip(all_retrieved_nodes, eval_data):
        metrics = calculate_metrics(nodes, item["relevant_pages"], k=top_k)
        all_metrics.append(metrics)
    
    df = pd.DataFrame(all_metrics)
    avg = df.mean().round(4)
    return avg

In [5]:
client = AsyncQdrantClient(url="http://127.0.0.1:6333",
                           prefer_grpc=True)
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
COLL_NAME = "wb_token_spl_550_75"
collect = client.get_collection(collection_name=COLL_NAME)

In [6]:
Settings.embed_model = OpenAILikeEmbedding(
    model_name="qwen3-embed",                    
    api_base="http://localhost:8081/v1",                      
)

try:
    req = requests.get(url="http://localhost:8081/health")
    if req.status_code == 200:
        print("✅ vLLM успешно подключён к LlamaIndex")
except Exception as e:
    print(f"Ошибка {e}")

✅ vLLM успешно подключён к LlamaIndex


In [7]:
# vector_store = QdrantVectorStore(collection_name=COLL_NAME,
#                                  client=client)

# # Settings.embed_model = TextEmbeddingsInference(model_name=EMBED_MODEL_NAME,
# #                                                base_url="http://localhost:8081")

# index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

In [8]:
eval_data_simple = []
with open("../context/docs_questions_qwen3_14b_awq.jsonl", "r") as f:
    lines = f.readlines()
    for line in lines:
        line = json.loads(line)
        eval_data_simple.append({
            "query": list(line.values())[0]["simple"],
            "relevant_pages": list(line.keys())[0].split("_")[1:]
        })


In [9]:
eval_data_medium = []
with open("../context/docs_questions_qwen3_14b_awq.jsonl", "r") as f:
    lines = f.readlines()
    for line in lines:
        line = json.loads(line)
        eval_data_medium.append({
            "query": list(line.values())[0]["medium"],
            "relevant_pages": list(line.keys())[0].split("_")[1:]
        })

In [10]:
eval_data_complex = []
with open("../context/docs_questions_qwen3_14b_awq.jsonl", "r") as f:
    lines = f.readlines()
    for line in lines:
        line = json.loads(line)
        eval_data_complex.append({
            "query": list(line.values())[0]["complex"],
            "relevant_pages": list(line.keys())[0].split("_")[1:]
        })

In [13]:
eval_full_data = []
with open("../context/docs_questions_qwen3_14b_awq.jsonl", "r") as f:
    lines = f.readlines()
    for line in lines:
        line = json.loads(line)
        eval_full_data.append({
            "query": list(line.values())[0]["simple"],
            "relevant_pages": list(line.keys())[0].split("_")[1:]
        })
        eval_full_data.append({
            "query": list(line.values())[0]["medium"],
            "relevant_pages": list(line.keys())[0].split("_")[1:]
        })
        eval_full_data.append({
            "query": list(line.values())[0]["complex"],
            "relevant_pages": list(line.keys())[0].split("_")[1:]
        })

In [32]:
print(len(eval_data_simple))
print(len(eval_data_medium))
print(len(eval_data_complex))
print(len(eval_full_data))

74
74
74
222


In [15]:
async def get_results(titles, eval_data):
    all_results = []

    for title in titles:
        # ВАЖНО: используем aclient для асинхронной работы
        vector_store = QdrantVectorStore(collection_name=title, aclient=client)
        index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

        for k in [1, 2, 3, 4, 5, 10]:
            print(f"Эксперимент: {title}, k={k}")
            
            # Ожидаем выполнения асинхронной оценки
            avg_metrics = await evaluate_collection(index, eval_data, top_k=k, retrieval_k=12)
            
            res_dict = avg_metrics.to_dict()
            res_dict['collection'] = title
            res_dict['k'] = k
            all_results.append(res_dict)

    df_results = pd.DataFrame(all_results)
    df_results.to_csv("../results/rag_experiments_results.csv", index=False)
    return df_results

In [16]:
# Добавляем await перед вызовом метода
collections_response = await client.get_collections()

# Теперь достаем имена
titles = [coll.name for coll in collections_response.collections]

In [29]:
await get_results(titles=titles, eval_data=eval_full_data)

Эксперимент: wb_token_spl_1753_75, k=1
Эксперимент: wb_token_spl_1753_75, k=2
Эксперимент: wb_token_spl_1753_75, k=3
Эксперимент: wb_token_spl_1753_75, k=4
Эксперимент: wb_token_spl_1753_75, k=5
Эксперимент: wb_token_spl_1753_75, k=10
Эксперимент: wb_token_spl_328_75, k=1
Эксперимент: wb_token_spl_328_75, k=2
Эксперимент: wb_token_spl_328_75, k=3
Эксперимент: wb_token_spl_328_75, k=4
Эксперимент: wb_token_spl_328_75, k=5
Эксперимент: wb_token_spl_328_75, k=10
Эксперимент: wb_token_spl_2287_75, k=1
Эксперимент: wb_token_spl_2287_75, k=2
Эксперимент: wb_token_spl_2287_75, k=3
Эксперимент: wb_token_spl_2287_75, k=4
Эксперимент: wb_token_spl_2287_75, k=5
Эксперимент: wb_token_spl_2287_75, k=10
Эксперимент: wb_token_spl_1575_75, k=1
Эксперимент: wb_token_spl_1575_75, k=2
Эксперимент: wb_token_spl_1575_75, k=3
Эксперимент: wb_token_spl_1575_75, k=4
Эксперимент: wb_token_spl_1575_75, k=5
Эксперимент: wb_token_spl_1575_75, k=10
Эксперимент: wb_token_spl_2465_75, k=1
Эксперимент: wb_token_spl_2

,precision@k,recall@k,f1@k,hit_rate@k,mrr@k,collection,k
0,0.5721,0.5721,0.5721,0.5721,0.5721,wb_token_spl_1753_75,1
1,0.4189,0.7027,0.5135,0.7027,0.6351,wb_token_spl_1753_75,2
2,0.3318,0.7568,0.4500,0.7568,0.6532,wb_token_spl_1753_75,3
3,0.2759,0.7928,0.3997,0.7928,0.6622,wb_token_spl_1753_75,4
4,0.2378,0.8378,0.3624,0.8378,0.6712,wb_token_spl_1753_75,5
...,...,...,...,...,...,...,...
97,0.4797,0.7613,0.5736,0.7613,0.6869,wb_token_spl_1040_75,2
98,0.3829,0.8243,0.5068,0.8243,0.7057,wb_token_spl_1040_75,3
99,0.3187,0.8694,0.4531,0.8694,0.7192,wb_token_spl_1040_75,4
100,0.2712,0.8694,0.4010,0.8694,0.7192,wb_token_spl_1040_75,5


In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
# 1. Загрузка и предобработка
df = pd.read_csv("../results/rag_experiments_results.csv")

def extract_params(row):
    # Ищем числа в формате ..._size_overlap
    match = re.search(r'(\d+)_(\d+)$', row['collection'])
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

df['chunk_size'], df['overlap'] = zip(*df.apply(extract_params, axis=1))
df = df.dropna(subset=['chunk_size']).sort_values('chunk_size')

# 2. Настройка параметров сетки
metrics = ['precision@k', 'recall@k', 'f1@k', 'hit_rate@k', 'mrr@k']
k_values = [1, 2, 3, 5, 10] # 5 колонок, как заказывали

fig, axes = plt.subplots(len(metrics), len(k_values), 
                         figsize=(25, 18), 
                         sharex='col', 
                         sharey='row')

plt.subplots_adjust(hspace=0.3, wspace=0.15)

# 3. Цикл построения
for row_idx, metric in enumerate(metrics):
    for col_idx, k_val in enumerate(k_values):
        ax = axes[row_idx, col_idx]
        
        # Фильтруем данные для конкретного k
        subset = df[df['k'] == k_val]
        
        # Строим график: зависимость метрики от размера чанка
        # hue='overlap' позволит увидеть разные линии, если у вас несколько вариантов перекрытия
        sns.lineplot(
            data=subset, 
            x='chunk_size', 
            y=metric, 
            marker='o', 
            ax=ax,
            hue='overlap',
            palette='viridis'
        )
        
        # Убираем легенду из всех графиков, кроме одного (чтобы не дублировать)
        if row_idx == 0 and col_idx == len(k_values)-1:
            ax.legend(title='Overlap', bbox_to_anchor=(1.05, 1))
        else:
            ax.get_legend().remove() if ax.get_legend() else None

        # Подписи только для краев сетки для чистоты
        if row_idx == 0:
            ax.set_title(f'K = {k_val}', fontsize=16, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(metric.replace('@k', '').upper(), fontsize=14, fontweight='bold')
        else:
            ax.set_ylabel('')
            
        ax.set_xlabel('Chunk Size' if row_idx == len(metrics)-1 else '')

# Сохранение
plt.savefig("../results/full_data_grid.png", dpi=300, bbox_inches='tight')
plt.show()